In [1]:
import boto3

In [2]:
redshift = boto3.client('redshift', region_name='us-east-1')

## Ejecutar este codigo solo 1 vez

In [3]:
try:
    response = redshift.create_cluster(
        ClusterIdentifier="bsg-cluster-3",
        NodeType="ra3.xlplus",
        ClusterType="single-node",
        MasterUsername="awsuser",
        MasterUserPassword="123.Abc*.*",
        DBName="dev",
        PubliclyAccessible=True
    )
    print("Cluster solicitado correctamente")
    print(response)

except Exception as e:
    print("Código:", e.response["Error"]["Code"])
    print("Mensaje:", e.response["Error"]["Message"])

Código: ClusterAlreadyExists
Mensaje: Cluster already exists


In [ ]:
print(response)

In [4]:
sts = boto3.client("sts")
identity = sts.get_caller_identity()

In [5]:
print(identity)

{'UserId': 'AROARPUURVY2LIJTMRD5A:user3033536=andres_frojas@hotmail.com', 'Account': '102317010484', 'Arn': 'arn:aws:sts::102317010484:assumed-role/voclabs/user3033536=andres_frojas@hotmail.com', 'ResponseMetadata': {'RequestId': '755b42d0-3001-4de2-8814-9e049c57008e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '755b42d0-3001-4de2-8814-9e049c57008e', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzc0NDA2MTk3OTg2OlI6SnkzcUVnMXA=', 'content-type': 'text/xml', 'content-length': '488', 'date': 'Wed, 25 Mar 2026 02:36:37 GMT'}, 'RetryAttempts': 0}}


## Pruebas de conexion al clustes de redshifts

In [9]:
import socket

socket.gethostbyname(
    "bsg-cluster-3.cdvovwnpoijz.us-east-1.redshift.amazonaws.com"
)

'3.209.113.182'

In [11]:
socket.create_connection(
    ("bsg-cluster-3.cdvovwnpoijz.us-east-1.redshift.amazonaws.com", 5439),
    timeout=5
)

<socket.socket fd=936, family=2, type=1, proto=0, laddr=('10.0.0.15', 55184), raddr=('3.209.113.182', 5439)>

## Conectar al cluster Redshift

In [6]:
import redshift_connector

In [12]:
conn = redshift_connector.connect(
    host='bsg-cluster-3.cdvovwnpoijz.us-east-1.redshift.amazonaws.com',
    port=5439,
    database='dev',
    user='awsuser',
    password='123.Abc*.*'
)

In [13]:
print(conn)

In [14]:
cursor = conn.cursor()

### Crear una tabla de ejemplo

In [ ]:

cursor.execute("""
CREATE TABLE ventas (
    id INT,
    fecha DATE,
    monto DECIMAL(10,2),
    categoria VARCHAR(50)
)
""")

conn.commit()

In [24]:
response = redshift.modify_cluster_iam_roles(
    ClusterIdentifier="bsg-cluster-3",
    AddIamRoles=[
        "arn:aws:iam::102317010484:role/LabRole"
    ]
)

### Copiar datos desde un CSV en S3 a la tabla (asumiendo el CSV no tiene cabecera y campos separados por coma)

In [28]:
try:
    cursor.execute("""
    COPY ventas
    FROM 's3://mi-bucket-datos-bsg/ventas2025.csv'
    IAM_ROLE 'arn:aws:iam::102317010484:role/myRedshiftRole'
    FORMAT AS CSV
    DELIMITER ','
    IGNOREHEADER 1;
    """)
    conn.commit()
    print("Carga exitosa")
except Exception as e:
    conn.rollback()
    print("Error:", e)

Carga exitosa


### Consultas SQL básicas: Ahora podemos ejecutar consultas SQL con el cursor. Por ejemplo, contar filas o hacer agregaciones:

In [29]:
cursor.execute("SELECT categoria, SUM(monto) as total FROM ventas GROUP BY 1;")
result = cursor.fetchall()
for row in result:
    print(row)

['Electrónica', Decimal('100.00')]
['Ropa', Decimal('130.00')]
['Hogar', Decimal('350.00')]
['Deportes', Decimal('285.00')]
['Libros', Decimal('170.00')]
['Juguetes', Decimal('230.00')]
['Salud', Decimal('160.00')]
['Alimentos', Decimal('120.00')]
['Automotriz', Decimal('240.00')]
['Viajes', Decimal('155.00')]
['Entretenimiento', Decimal('225.00')]
['Música', Decimal('170.00')]
['Arte', Decimal('85.00')]
['Oficina', Decimal('75.00')]
['Salón de Belleza', Decimal('140.00')]
['Restaurantes', Decimal('65.00')]
['Servicios', Decimal('115.00')]
['Electrodomésticos', Decimal('55.00')]
['Accesorios', Decimal('125.00')]
['Regalos', Decimal('105.00')]
